# Notebook 03 — Custom Transformer: Implementation and Correctness Tests

**Track A · Soccer Event Transformer · Identity × Context project**

| Item | Content |
|---|---|
| **Research question (project level)** | When identity and context point to different actions, what does a trained model do? Track A trains and freezes the model; this notebook implements it. |
| **Purpose of this notebook** | Implement the decoder-only Transformer from explicit tensor operations, and test — before any training — that the code computes the intended mathematics. |
| **Architecture** | Decoder-only, causal attention, 4 layers, d_model 128, 4 heads of 32, d_ff 512, context 32, pre-LayerNorm, GELU, no dropout, learned positions, factorised event representation, linear output head. Depth, head count and context length are varied later, in notebook 05. |


In [1]:
# ---------------------------------------------------------------------------
# Environment setup (identical at the top of every Track A notebook)
# ---------------------------------------------------------------------------
import os
import sys
import json
import time
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

# PROJECT_DIR holds code (src/), data artifacts, runs, results and the release.
PROJECT_DIR = Path(os.environ.get("TRACK_A_DIR", "/content/drive/MyDrive/soccer_mi/track_a"))
# DATASET_PATH is the prepared research dataset (one row per on-ball event).
DATASET_PATH = Path(os.environ.get("TRACK_A_DATASET", str(PROJECT_DIR / "data" / "cleaned_dataset.parquet")))
# RUN_MODE = "full"  -> real research runs (use this for every reported number)
# RUN_MODE = "smoke" -> tiny code check only; outputs go to PROJECT_DIR/smoke and are NOT results
RUN_MODE = os.environ.get("TRACK_A_RUN_MODE", "full")
assert RUN_MODE in ("full", "smoke")

PROJECT_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(PROJECT_DIR)
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))
(PROJECT_DIR / "src").mkdir(exist_ok=True)
(PROJECT_DIR / "tests").mkdir(exist_ok=True)

# Data-audit artifacts are cheap and always produced in full mode.
ARTIFACTS_DIR = PROJECT_DIR / "artifacts"
# Training runs, results and figures are separated by run mode so a smoke check can never
# overwrite or be confused with a research result.
OUTPUT_DIR = PROJECT_DIR if RUN_MODE == "full" else PROJECT_DIR / "smoke"
RUNS_DIR = OUTPUT_DIR / "runs"
RESULTS_DIR = OUTPUT_DIR / "results"
FIGURES_DIR = OUTPUT_DIR / "figures"
for folder in [ARTIFACTS_DIR, RUNS_DIR, RESULTS_DIR, FIGURES_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Colab:", IN_COLAB, "| RUN_MODE:", RUN_MODE)
print("PROJECT_DIR:", PROJECT_DIR)
print("DATASET_PATH:", DATASET_PATH, "| exists:", DATASET_PATH.exists())

Mounted at /content/drive
Colab: True | RUN_MODE: full
PROJECT_DIR: /content/drive/MyDrive/soccer_mi/track_a
DATASET_PATH: /content/drive/MyDrive/soccer_mi/track_a/data/cleaned_dataset.parquet | exists: True


## 1. How the model works — the terms

| Term | Meaning here |
|---|---|
| **Token** | One event. A window of 32 events is a sequence of 32 tokens. |
| **Embedding** | A learned vector looked up by an integer index (for example, the vector for zone 52). |
| **Factorised event representation** | Each event is described by several *factors* (player, team, role, competition, space, time, game state, history). Each factor is turned into its own 128-number vector, and the event's input vector is the **sum** of these factor vectors plus a position vector. Because the factors are added rather than mixed, any single factor can later be removed or replaced without touching the others. |
| **Positional embedding** | A learned vector for each position 0–31 in the window, added so the model knows the order of events. |
| **Residual stream** | The running 128-number vector at each position. It starts as the event's input vector; every attention and MLP sub-layer *adds* its output to it. The final prediction is read from it. |
| **Self-attention** | Lets each position gather information from other positions. Each position produces a **query** (what the position looks for), a **key** (what the position contains) and a **value** (what the position passes on). The score between two positions is query · key; scores are turned into weights with softmax, and the output is the weighted sum of values. |
| **Causal attention** | A position may attend only to itself and **earlier** positions. Scores for later positions are set to −∞ before softmax, so their weight is exactly zero. This is what prevents the model from seeing the future. |
| **Head** | One independent attention computation on a 32-number slice. With 4 heads, a layer can follow 4 different "where to look" rules at once. |
| **MLP** | A two-layer network applied to each position separately: 128 → 512 → 128 with a GELU non-linearity. It transforms information already present at a position. |
| **GELU** | A smooth activation function: negative inputs are mostly suppressed, positive inputs pass through. |
| **LayerNorm** | Rescales a vector to mean 0 and variance 1, then applies a learned scale and shift. It keeps numbers in a stable range. |
| **Pre-LayerNorm** | LayerNorm is applied to the *input* of each sub-layer, not to the residual stream itself. The residual stream therefore stays an exact sum of everything written into it. |
| **Logits / unembedding** | After a final LayerNorm, a linear layer (the *unembedding*) maps the 128-number vector to 15 logits, one per action class. |
| **Cross-entropy loss** | The training objective: −log of the probability given to the true action, averaged over positions. Identical to NLL. |
| **Parameter count** | The number of trainable numbers in the model. |

## 2. Batching: turning windows into model inputs

A window is stored as 32 row numbers into the event table. `EncodedData` keeps every input feature as one long tensor (one value per event) on the GPU; building a batch is simply looking up the rows of each window. No data is copied into per-window files, so every batch can be traced back to specific events.

In [2]:
%%writefile src/batching.py
"""
src/batching.py — build model inputs from windows of event row numbers (notebook 03).
"""
import numpy as np
import torch


class EncodedData:
    """All model inputs for all events, stored as tensors on one device.

    features : dict name -> numpy array with one value per event (from data.encode_features)
    targets  : numpy array of action ids, one per event
    """

    def __init__(self, features, targets, device):
        self.device = device
        self.tensors = {}
        for name, values in features.items():
            self.tensors[name] = torch.as_tensor(np.asarray(values), device=device)
        self.targets = torch.as_tensor(np.asarray(targets), device=device)

    def batch(self, window_rows):
        """window_rows: LongTensor [batch, positions] of event row numbers.

        Returns (inputs, targets):
          inputs  dict name -> tensor [batch, positions]
          targets tensor [batch, positions] of action ids
        """
        inputs = {}
        for name, tensor in self.tensors.items():
            inputs[name] = tensor[window_rows]
        targets = self.targets[window_rows]
        return inputs, targets

Overwriting src/batching.py


## 3. The model, block by block

The model is written from basic PyTorch layers (`nn.Linear`, `nn.Embedding`, `nn.LayerNorm`) and explicit tensor operations. No high-level Transformer class is used, so every step of the computation is visible.

### 3.1 Configuration and the factorised event embedding

`use_groups` lists which factor groups the model receives. The full model uses all of them; the restricted baselines of notebook 06 (context-only, identity-only, role-only) are the *same* model with some groups switched off, so their comparison changes only the information, not the architecture.

**Identity tables use index 0 as a fixed zero vector** (`padding_idx=0`): it is initialised to zeros and receives no gradient, so it never learns a meaning.

**Continuous inputs** (x, y, time gap, possession age) are mapped to 128 numbers by a linear layer **without a bias term**. A bias would add a constant vector that belongs to no input, which would blur the separation between factors.

In [3]:
%%writefile src/model.py
"""
src/model.py — custom decoder-only Transformer for next-action prediction (notebook 03).

Every step is written with explicit tensor operations so the computation can be read and
verified. Shapes use: B = batch size, T = positions (context length), D = d_model,
H = number of heads, E = head dimension (D / H), F = d_ff, C = number of action classes.
"""
import math
from dataclasses import dataclass, field, asdict

import torch
import torch.nn as nn
import torch.nn.functional as F

IDENTITY_FEATURES = ["player", "team", "role", "competition"]

# Categorical inputs belonging to each factor group.
FACTOR_GROUPS = {
    "player": ["player"],
    "team": ["team"],
    "role": ["role"],
    "competition": ["competition"],
    "space": ["zone"],
    "time": ["period", "minute_bin"],
    "state": ["score_diff", "poss_len", "possession_flag"],
    "history": ["prev_action", "prev_outcome", "prev_end_zone", "same_player_as_prev"],
}
# Continuous inputs belonging to each factor group (projected by a bias-free linear layer).
CONTINUOUS_GROUPS = {
    "space": ["x_norm", "y_norm"],
    "time": ["dt", "t_poss"],
}
ALL_GROUPS = ["player", "team", "role", "competition", "space", "time", "state", "history"]


@dataclass
class ModelConfig:
    vocabulary_sizes: dict
    n_layers: int = 4
    d_model: int = 128
    n_heads: int = 4
    d_ff: int = 512
    context_length: int = 32
    n_actions: int = 15
    use_groups: list = field(default_factory=lambda: list(ALL_GROUPS))
    use_position: bool = True
    init_std: float = 0.02

    def to_dict(self):
        return asdict(self)


class FactorizedEventEmbedding(nn.Module):
    """Input vector of each event = sum of one vector per factor group + a position vector."""

    def __init__(self, config):
        super().__init__()
        self.config = config
        self.groups = list(config.use_groups)
        self.embeddings = nn.ModuleDict()
        for group in self.groups:
            for feature in FACTOR_GROUPS[group]:
                table_size = config.vocabulary_sizes[feature]
                if feature in IDENTITY_FEATURES:
                    # index 0 = "no dedicated identity": a fixed zero vector, never trained
                    self.embeddings[feature] = nn.Embedding(table_size, config.d_model, padding_idx=0)
                else:
                    self.embeddings[feature] = nn.Embedding(table_size, config.d_model)
        self.continuous_projections = nn.ModuleDict()
        for group, names in CONTINUOUS_GROUPS.items():
            if group in self.groups:
                self.continuous_projections[group] = nn.Linear(len(names), config.d_model, bias=False)
        if config.use_position:
            self.position_embedding = nn.Embedding(config.context_length, config.d_model)

    def forward(self, inputs):
        """inputs: dict name -> tensor [B, T]. Returns (total [B, T, D], factor_vectors dict)."""
        factor_vectors = {}
        for group in self.groups:
            group_vector = None
            for feature in FACTOR_GROUPS[group]:
                vector = self.embeddings[feature](inputs[feature])            # [B, T, D]
                if group_vector is None:
                    group_vector = vector
                else:
                    group_vector = group_vector + vector
            if group in self.continuous_projections:
                values = torch.stack([inputs[name] for name in CONTINUOUS_GROUPS[group]], dim=-1)   # [B, T, 2]
                group_vector = group_vector + self.continuous_projections[group](values)
            factor_vectors[group] = group_vector

        any_input = inputs["zone"]
        batch_size = any_input.shape[0]
        n_positions = any_input.shape[1]
        if self.config.use_position:
            positions = torch.arange(n_positions, device=any_input.device)
            position_vectors = self.position_embedding(positions)             # [T, D]
            factor_vectors["position"] = position_vectors.unsqueeze(0).expand(batch_size, n_positions, -1)

        total = None
        for name in factor_vectors:
            if total is None:
                total = factor_vectors[name]
            else:
                total = total + factor_vectors[name]
        return total, factor_vectors

Overwriting src/model.py


### 3.2 Causal multi-head self-attention

Step by step, for input `x` of shape [B, T, 128]:

1. **Q, K, V projections**: three linear layers produce queries, keys and values, each [B, T, 128].
2. **Split into heads**: reshape to [B, H, T, 32] — each of the 4 heads gets its own 32-number slice.
3. **Scores**: `scores = Q · Kᵀ / √32`, shape [B, H, T, T]. Entry (i, j) says how much position i wants to look at position j. Dividing by √32 keeps the scores in a range where softmax is not saturated.
4. **Causal mask**: entries with j > i (the future) are set to −∞.
5. **Attention probabilities**: softmax over j. Each row sums to 1 and future entries are exactly 0.
6. **Head outputs**: `probabilities · V`, shape [B, H, T, 32].
7. **Concatenate heads** back to [B, T, 128] and apply the **output projection** `W_O`.

In [4]:
%%writefile -a src/model.py

class CausalSelfAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        assert config.d_model % config.n_heads == 0, "d_model must be divisible by n_heads"
        self.n_heads = config.n_heads
        self.d_head = config.d_model // config.n_heads
        self.W_Q = nn.Linear(config.d_model, config.d_model)
        self.W_K = nn.Linear(config.d_model, config.d_model)
        self.W_V = nn.Linear(config.d_model, config.d_model)
        self.W_O = nn.Linear(config.d_model, config.d_model)
        # causal_mask[i, j] is True when position i may look at position j (j <= i)
        causal_mask = torch.tril(torch.ones(config.context_length, config.context_length, dtype=torch.bool))
        self.register_buffer("causal_mask", causal_mask, persistent=False)

    def split_heads(self, tensor):
        """[B, T, D] -> [B, H, T, E]"""
        batch_size, n_positions, _ = tensor.shape
        tensor = tensor.view(batch_size, n_positions, self.n_heads, self.d_head)
        return tensor.transpose(1, 2)

    def forward(self, x, return_probabilities=False):
        batch_size, n_positions, d_model = x.shape

        queries = self.split_heads(self.W_Q(x))                      # [B, H, T, E]
        keys = self.split_heads(self.W_K(x))                         # [B, H, T, E]
        values = self.split_heads(self.W_V(x))                       # [B, H, T, E]

        scores = torch.matmul(queries, keys.transpose(-2, -1))       # [B, H, T, T]
        scores = scores / math.sqrt(self.d_head)

        mask = self.causal_mask[:n_positions, :n_positions]
        scores = scores.masked_fill(~mask, float("-inf"))            # future positions -> -inf

        probabilities = torch.softmax(scores, dim=-1)                # rows sum to 1
        head_outputs = torch.matmul(probabilities, values)           # [B, H, T, E]

        head_outputs = head_outputs.transpose(1, 2)                  # [B, T, H, E]
        concatenated = head_outputs.reshape(batch_size, n_positions, d_model)
        output = self.W_O(concatenated)                              # [B, T, D]

        if return_probabilities:
            return output, probabilities
        return output

Appending to src/model.py


### 3.3 MLP, Transformer block (pre-LayerNorm) and the full model

One block computes:

```
attention_input  = LayerNorm_1(residual)
residual_mid     = residual + Attention(attention_input)        # residual addition 1
mlp_input        = LayerNorm_2(residual_mid)
residual_post    = residual_mid + MLP(mlp_input)                # residual addition 2
```

The full model: factorised embedding → 4 blocks → final LayerNorm → unembedding (128 → 15 logits).

**Initialisation** (engineering choice): weights drawn from a normal distribution with standard deviation 0.02; biases zero. The two layers that write into the residual stream (attention output projection and the MLP's second layer) use 0.02 / √(2 · layers), so that the residual stream does not grow as more blocks add to it. Identity rows at index 0 are reset to zero after initialisation.

`return_internals=True` returns intermediate tensors. It is used **only by the correctness tests below**; it is not intended as interpretability instrumentation.

In [5]:
%%writefile -a src/model.py

class MLP(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.input_layer = nn.Linear(config.d_model, config.d_ff)     # D -> F
        self.output_layer = nn.Linear(config.d_ff, config.d_model)    # F -> D

    def forward(self, x):
        hidden = self.input_layer(x)
        hidden = F.gelu(hidden)
        return self.output_layer(hidden)


class TransformerBlock(nn.Module):
    """Pre-LayerNorm block: residual += Attention(LN(residual)); residual += MLP(LN(residual))."""

    def __init__(self, config):
        super().__init__()
        self.ln_attention = nn.LayerNorm(config.d_model)
        self.attention = CausalSelfAttention(config)
        self.ln_mlp = nn.LayerNorm(config.d_model)
        self.mlp = MLP(config)

    def forward(self, residual, internals=None):
        attention_input = self.ln_attention(residual)
        if internals is not None:
            attention_output, probabilities = self.attention(attention_input, return_probabilities=True)
        else:
            attention_output = self.attention(attention_input)
        residual_mid = residual + attention_output

        mlp_input = self.ln_mlp(residual_mid)
        mlp_output = self.mlp(mlp_input)
        residual_post = residual_mid + mlp_output

        if internals is not None:
            internals.append({
                "resid_pre": residual, "attention_input": attention_input,
                "attention_probabilities": probabilities, "attention_output": attention_output,
                "resid_mid": residual_mid, "mlp_input": mlp_input, "mlp_output": mlp_output,
                "resid_post": residual_post,
            })
        return residual_post


class EventTransformer(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.embedding = FactorizedEventEmbedding(config)
        self.blocks = nn.ModuleList()
        for _ in range(config.n_layers):
            self.blocks.append(TransformerBlock(config))
        self.ln_final = nn.LayerNorm(config.d_model)
        self.unembed = nn.Linear(config.d_model, config.n_actions)    # D -> C
        self.initialise_weights()

    def initialise_weights(self):
        std = self.config.init_std
        for module in self.modules():
            if isinstance(module, nn.Linear):
                nn.init.normal_(module.weight, mean=0.0, std=std)
                if module.bias is not None:
                    nn.init.zeros_(module.bias)
            elif isinstance(module, nn.Embedding):
                nn.init.normal_(module.weight, mean=0.0, std=std)
                if module.padding_idx is not None:
                    with torch.no_grad():
                        module.weight[module.padding_idx].zero_()
        # layers that write into the residual stream get a smaller scale
        residual_std = std / math.sqrt(2 * self.config.n_layers)
        for block in self.blocks:
            nn.init.normal_(block.attention.W_O.weight, mean=0.0, std=residual_std)
            nn.init.normal_(block.mlp.output_layer.weight, mean=0.0, std=residual_std)

    def forward(self, inputs, return_internals=False):
        """inputs: dict name -> [B, T]. Returns logits [B, T, C] (and internals if requested)."""
        residual, factor_vectors = self.embedding(inputs)
        internals = [] if return_internals else None
        for block in self.blocks:
            residual = block(residual, internals)
        final = self.ln_final(residual)
        logits = self.unembed(final)
        if return_internals:
            return logits, {"embedding": residual if self.config.n_layers == 0 else internals[0]["resid_pre"],
                            "factor_vectors": factor_vectors, "blocks": internals,
                            "final_residual": residual, "final_normalised": final}
        return logits


def compute_loss(logits, targets, valid_mask=None):
    """Mean cross-entropy over valid positions. logits [B, T, C], targets [B, T]."""
    n_classes = logits.shape[-1]
    flat_logits = logits.reshape(-1, n_classes)
    flat_targets = targets.reshape(-1)
    per_position = F.cross_entropy(flat_logits, flat_targets, reduction="none")
    if valid_mask is None:
        return per_position.mean()
    flat_mask = valid_mask.reshape(-1).float()
    return (per_position * flat_mask).sum() / flat_mask.sum()


def count_parameters(model):
    """Parameter counts by component. Fixed zero rows (identity index 0) are reported separately."""
    counts = {}
    counts["embeddings (tables + projections + positions)"] = sum(p.numel() for p in model.embedding.parameters())
    counts["transformer blocks"] = sum(p.numel() for p in model.blocks.parameters())
    counts["final LayerNorm"] = sum(p.numel() for p in model.ln_final.parameters())
    counts["unembedding"] = sum(p.numel() for p in model.unembed.parameters())
    total = sum(p.numel() for p in model.parameters())
    fixed_zero = 0
    for module in model.embedding.embeddings.values():
        if module.padding_idx is not None:
            fixed_zero += module.embedding_dim
    counts["total"] = total
    counts["fixed zero rows (never trained)"] = fixed_zero
    counts["trainable"] = total - fixed_zero
    return counts

Appending to src/model.py


In [6]:
import numpy as np
import pandas as pd
import torch
import importlib
import src.data as data
import src.batching as batching
import src.model as model_module
for module in [data, batching, model_module]:
    importlib.reload(module)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE, "| torch", torch.__version__)

events = pd.read_parquet(ARTIFACTS_DIR / "events_prepared.parquet")
encoding = data.load_encoding(ARTIFACTS_DIR / "encodings" / "T100")
features, vocabulary_sizes = data.encode_features(events, encoding, possession_features="pre_action")
targets = events["action_id"].to_numpy().astype("int64")
encoded = batching.EncodedData(features, targets, DEVICE)
val_windows = data.build_windows(events, 32, 8, split="val")

def make_batch(n_windows=8, first_window=0):
    rows = torch.as_tensor(val_windows[first_window:first_window + n_windows], device=DEVICE)
    return encoded.batch(rows)

def make_model(seed=0, **overrides):
    torch.manual_seed(seed)
    config = model_module.ModelConfig(vocabulary_sizes=vocabulary_sizes, **overrides)
    return model_module.EventTransformer(config).to(DEVICE)

model = make_model()
inputs, batch_targets = make_batch()
logits, internals = model(inputs, return_internals=True)
print("logits shape:", tuple(logits.shape))

Device: cpu | torch 2.11.0+cpu
logits shape: (8, 32, 15)


### 3.4 Tensor shapes through the model

The table traces one real batch (8 validation windows) through the model and prints the shape of every intermediate tensor. It is the concrete version of the equations above.

In [7]:
first_block = internals["blocks"][0]
shape_rows = [
    ("one input feature, e.g. zone", tuple(inputs["zone"].shape), "[B, T]"),
    ("player factor vector", tuple(internals["factor_vectors"]["player"].shape), "[B, T, D]"),
    ("event input vector (sum of factors + position)", tuple(internals["embedding"].shape), "[B, T, D]"),
    ("attention input (after LayerNorm 1)", tuple(first_block["attention_input"].shape), "[B, T, D]"),
    ("attention probabilities", tuple(first_block["attention_probabilities"].shape), "[B, H, T, T]"),
    ("attention output (after W_O)", tuple(first_block["attention_output"].shape), "[B, T, D]"),
    ("residual after attention", tuple(first_block["resid_mid"].shape), "[B, T, D]"),
    ("MLP output", tuple(first_block["mlp_output"].shape), "[B, T, D]"),
    ("residual after block", tuple(first_block["resid_post"].shape), "[B, T, D]"),
    ("final normalised residual", tuple(internals["final_normalised"].shape), "[B, T, D]"),
    ("logits", tuple(logits.shape), "[B, T, C]"),
]
pd.DataFrame(shape_rows, columns=["tensor", "shape", "meaning (B=8, T=32, D=128, H=4, C=15)"])

,tensor,shape,"meaning (B=8, T=32, D=128, H=4, C=15)"
0,"one input feature, e.g. zone","(8, 32)","[B, T]"
1,player factor vector,"(8, 32, 128)","[B, T, D]"
2,event input vector (sum of factors + position),"(8, 32, 128)","[B, T, D]"
3,attention input (after LayerNorm 1),"(8, 32, 128)","[B, T, D]"
4,attention probabilities,"(8, 4, 32, 32)","[B, H, T, T]"
5,attention output (after W_O),"(8, 32, 128)","[B, T, D]"
6,residual after attention,"(8, 32, 128)","[B, T, D]"
7,MLP output,"(8, 32, 128)","[B, T, D]"
8,residual after block,"(8, 32, 128)","[B, T, D]"
9,final normalised residual,"(8, 32, 128)","[B, T, D]"


## 4. Parameter counts

Exact counts for the approved configuration under both player thresholds, and for the restricted baselines that reuse the same architecture. "Fixed zero rows" are the index-0 identity rows: they exist in memory but are never trained.

In [8]:
encoding_T300 = data.load_encoding(ARTIFACTS_DIR / "encodings" / "T300")
_, vocabulary_sizes_T300 = data.encode_features(events, encoding_T300)

variants = {
    "full model, T100": (vocabulary_sizes, model_module.ALL_GROUPS),
    "full model, T300": (vocabulary_sizes_T300, model_module.ALL_GROUPS),
    "B5 context-only (no player/team/role)": (vocabulary_sizes, ["competition", "space", "time", "state", "history"]),
    "B6 identity-only (player/team/role)": (vocabulary_sizes, ["player", "team", "role"]),
    "B7 role-only (no player/team)": (vocabulary_sizes, ["role", "competition", "space", "time", "state", "history"]),
}
count_rows = {}
for name, (sizes, groups) in variants.items():
    torch.manual_seed(0)
    variant_model = model_module.EventTransformer(model_module.ModelConfig(vocabulary_sizes=sizes, use_groups=list(groups)))
    count_rows[name] = model_module.count_parameters(variant_model)
parameter_table = pd.DataFrame(count_rows).T
parameter_table.to_csv(RESULTS_DIR / "03_parameter_counts.csv")
parameter_table

,embeddings (tables + projections + positions),transformer blocks,final LayerNorm,unembedding,total,fixed zero rows (never trained),trainable
"full model, T100",289536,793088,256,1935,1084815,512,1084303
"full model, T300",117888,793088,256,1935,913167,512,912655
B5 context-only (no player/team/role),42240,793088,256,1935,837519,128,837391
B6 identity-only (player/team/role),251392,793088,256,1935,1046671,384,1046287
B7 role-only (no player/team),45568,793088,256,1935,840847,256,840591


**RESULT.** Exact parameter counts are listed per component.

**OBSERVATION.** The four Transformer blocks contribute 793,088 parameters in every variant. The full model has 1,084,815 parameters with T100 (1,084,303 trainable) and 913,167 with T300 (912,655 trainable); the difference is entirely the player table.

**INTERPRETATION.** The planning figure of "≈1.15M parameters" does not match either configuration. The exact numbers above replace it. At T100 roughly one fifth of all parameters are player vectors. The counts do not include a play-pattern embedding, because that input is not part of the model (notebook 01, Section 10).

**LIMITATION.** Parameter count is a size measure, not a capacity measure: most player vectors are updated only when that player appears.

## 5. Correctness tests

A model that runs without errors is not necessarily correct: a wrong mask or a misplaced LayerNorm still trains, just on a different computation. The tests below check the **mathematics**, before any training. The tests are kept in a separate module (`tests/test_model.py`).

| # | Test | What would go wrong without it |
|---|---|---|
| 1 | Shapes of every intermediate tensor | silent broadcasting errors |
| 2 | Causal mask: changing future events does not change earlier outputs | the model could see the answer |
| 3 | Attention equals PyTorch's reference `scaled_dot_product_attention` | wrong scaling, masking or head split |
| 4 | Attention probabilities: rows sum to 1, future weights exactly 0 | invalid weighting |
| 5 | Residual additions recomputed from their parts | a sub-layer output not added (or added twice) |
| 6 | LayerNorm placement (pre-LN) | normalisation applied to the wrong tensor |
| 7 | Positional embeddings affect the output and receive gradient | the model cannot tell event order |
| 8 | Output has 15 classes | wrong target vocabulary (e.g. END included) |
| 9 | Determinism: same seed → identical parameters, outputs and short training run | results not reproducible |
| 10 | Gradient flow: every parameter gets a finite gradient; index-0 rows get none | dead or exploding parts |
| 11 | Loss equals manual −log softmax; initial predictions are near-uniform (entropy ≈ ln 15) | wrong loss or over-confident initialisation |
| 12 | Masked positions do not affect the loss | padding would leak into training |
| 13 | The model can memorise one small batch | the model cannot learn at all |
| 14 | Trained on shuffled labels, it cannot beat the class prior on validation | a hidden leak of the target |
| 15 | Target-side columns cannot change the inputs | leakage through feature building |
| 16 | Identity index 0 stays exactly zero after optimiser steps | index 0 acquiring a hidden meaning |

In [9]:
%%writefile tests/test_model.py
"""
tests/test_model.py — mathematical correctness tests for src/model.py (notebook 03).

Every test takes a `context` dict with:
  make_model(seed, **overrides) -> EventTransformer on the right device
  make_batch(n_windows, first_window) -> (inputs, targets)
and returns (passed: bool, detail: str).
"""
import math
import numpy as np
import torch
import torch.nn.functional as F

import src.model as model_module


def test_shapes(context):
    model = context["make_model"](0)
    inputs, targets = context["make_batch"](8, 0)
    logits, internals = model(inputs, return_internals=True)
    batch_size, n_positions = targets.shape
    config = model.config
    expected = {
        "logits": (batch_size, n_positions, config.n_actions),
        "attention_probabilities": (batch_size, config.n_heads, n_positions, n_positions),
        "resid_post": (batch_size, n_positions, config.d_model),
    }
    observed = {
        "logits": tuple(logits.shape),
        "attention_probabilities": tuple(internals["blocks"][0]["attention_probabilities"].shape),
        "resid_post": tuple(internals["blocks"][-1]["resid_post"].shape),
    }
    passed = expected == observed and len(internals["blocks"]) == config.n_layers
    return passed, f"observed {observed}"


def test_causal_mask(context):
    model = context["make_model"](0)
    inputs, _ = context["make_batch"](8, 0)
    other_inputs, _ = context["make_batch"](8, 100)
    cut = 10
    changed = {}
    for name in inputs:
        tensor = inputs[name].clone()
        tensor[:, cut + 1:] = other_inputs[name][:, cut + 1:]      # replace every future event
        changed[name] = tensor
    with torch.no_grad():
        original = model(inputs)
        perturbed = model(changed)
    difference_before = (original[:, :cut + 1] - perturbed[:, :cut + 1]).abs().max().item()
    difference_after = (original[:, cut + 1:] - perturbed[:, cut + 1:]).abs().max().item()
    passed = difference_before == 0.0 and difference_after > 0.0
    return passed, f"max change at positions <= {cut}: {difference_before}; after: {difference_after:.3g}"


def test_attention_matches_reference(context):
    model = context["make_model"](0)
    attention = model.blocks[0].attention
    torch.manual_seed(1)
    x = torch.randn(4, 32, model.config.d_model, device=next(model.parameters()).device)
    with torch.no_grad():
        ours = attention(x)
        queries = attention.split_heads(attention.W_Q(x))
        keys = attention.split_heads(attention.W_K(x))
        values = attention.split_heads(attention.W_V(x))
        reference_heads = F.scaled_dot_product_attention(queries, keys, values, is_causal=True)
        reference = attention.W_O(reference_heads.transpose(1, 2).reshape(4, 32, model.config.d_model))
    max_difference = (ours - reference).abs().max().item()
    return max_difference < 1e-5, f"max |ours - reference| = {max_difference:.2e}"


def test_attention_probabilities(context):
    model = context["make_model"](0)
    inputs, _ = context["make_batch"](8, 0)
    with torch.no_grad():
        _, internals = model(inputs, return_internals=True)
    worst_row_sum = 0.0
    worst_future = 0.0
    for block in internals["blocks"]:
        probabilities = block["attention_probabilities"]
        worst_row_sum = max(worst_row_sum, (probabilities.sum(-1) - 1).abs().max().item())
        future = torch.triu(torch.ones(32, 32, dtype=torch.bool, device=probabilities.device), diagonal=1)
        worst_future = max(worst_future, probabilities[..., future].abs().max().item())
    passed = worst_row_sum < 1e-5 and worst_future == 0.0
    return passed, f"max |row sum - 1| = {worst_row_sum:.2e}; max future weight = {worst_future}"


def test_residual_additions(context):
    model = context["make_model"](0)
    inputs, _ = context["make_batch"](8, 0)
    with torch.no_grad():
        _, internals = model(inputs, return_internals=True)
    errors = []
    factor_sum = sum(internals["factor_vectors"].values())
    errors.append((factor_sum - internals["blocks"][0]["resid_pre"]).abs().max().item())
    previous_post = None
    for block in internals["blocks"]:
        errors.append((block["resid_pre"] + block["attention_output"] - block["resid_mid"]).abs().max().item())
        errors.append((block["resid_mid"] + block["mlp_output"] - block["resid_post"]).abs().max().item())
        if previous_post is not None:
            errors.append((previous_post - block["resid_pre"]).abs().max().item())
        previous_post = block["resid_post"]
    worst = max(errors)
    return worst < 1e-5, f"max reconstruction error = {worst:.2e}"


def test_layernorm_placement(context):
    model = context["make_model"](0)
    inputs, _ = context["make_batch"](8, 0)
    with torch.no_grad():
        _, internals = model(inputs, return_internals=True)
    errors = []
    for layer_number, block in enumerate(internals["blocks"]):
        module_block = model.blocks[layer_number]
        errors.append((module_block.ln_attention(block["resid_pre"]) - block["attention_input"]).abs().max().item())
        errors.append((module_block.ln_mlp(block["resid_mid"]) - block["mlp_input"]).abs().max().item())
    # explicit LayerNorm formula: (x - mean) / sqrt(variance + eps) * scale + shift
    residual = internals["blocks"][0]["resid_pre"]
    layer_norm = model.blocks[0].ln_attention
    mean = residual.mean(dim=-1, keepdim=True)
    variance = residual.var(dim=-1, unbiased=False, keepdim=True)
    manual = (residual - mean) / torch.sqrt(variance + layer_norm.eps) * layer_norm.weight + layer_norm.bias
    formula_error = (manual - internals["blocks"][0]["attention_input"]).abs().max().item()
    final_error = (model.ln_final(internals["final_residual"]) - internals["final_normalised"]).abs().max().item()
    passed = max(errors) < 1e-6 and final_error < 1e-6 and formula_error < 1e-5
    return passed, (f"sub-layer inputs equal LN(residual): error {max(errors):.1e}; "
                    f"explicit formula (x-mean)/sqrt(var+eps)*scale+shift: error {formula_error:.1e}")


def test_positional_embeddings(context):
    model = context["make_model"](0)
    inputs, targets = context["make_batch"](8, 0)
    with torch.no_grad():
        original = model(inputs)
        saved = model.embedding.position_embedding.weight.clone()
        model.embedding.position_embedding.weight.zero_()
        without_positions = model(inputs)
        model.embedding.position_embedding.weight.copy_(saved)
    output_change = (original - without_positions).abs().max().item()
    loss = model_module.compute_loss(model(inputs), targets)
    loss.backward()
    gradient_norm = model.embedding.position_embedding.weight.grad.norm().item()
    passed = output_change > 0 and gradient_norm > 0
    return passed, f"output change when positions removed {output_change:.3g}; position gradient norm {gradient_norm:.3g}"


def test_output_dimension(context):
    model = context["make_model"](0)
    inputs, _ = context["make_batch"](2, 0)
    with torch.no_grad():
        logits = model(inputs)
    return logits.shape[-1] == 15, f"classes = {logits.shape[-1]} (15 actions; END is not a target)"


def test_determinism(context):
    model_a = context["make_model"](123)
    model_b = context["make_model"](123)
    parameters_identical = all(torch.equal(a, b) for a, b in zip(model_a.parameters(), model_b.parameters()))
    inputs, targets = context["make_batch"](8, 0)
    with torch.no_grad():
        outputs_identical = torch.equal(model_a(inputs), model_a(inputs))

    def short_run(seed):
        model = context["make_model"](seed)
        optimiser = torch.optim.AdamW(model.parameters(), lr=1e-3)
        losses = []
        for step in range(10):
            step_inputs, step_targets = context["make_batch"](8, step * 8)
            loss = model_module.compute_loss(model(step_inputs), step_targets)
            optimiser.zero_grad()
            loss.backward()
            optimiser.step()
            losses.append(loss.item())
        return losses

    run_a = short_run(7)
    run_b = short_run(7)
    passed = parameters_identical and outputs_identical and run_a == run_b
    return passed, f"params identical {parameters_identical}; outputs identical {outputs_identical}; 10-step losses identical {run_a == run_b}"


def test_gradient_flow(context):
    model = context["make_model"](0)
    inputs, targets = context["make_batch"](16, 0)
    loss = model_module.compute_loss(model(inputs), targets)
    loss.backward()
    problems = []
    for name, parameter in model.named_parameters():
        if parameter.grad is None:
            problems.append(f"{name}: no gradient")
            continue
        if not torch.isfinite(parameter.grad).all():
            problems.append(f"{name}: non-finite gradient")
            continue
        gradient = parameter.grad
        if name.endswith("weight") and "embeddings" in name:
            feature = name.split(".")[2]
            if feature in model_module.IDENTITY_FEATURES:
                if gradient[0].abs().max().item() != 0.0:
                    problems.append(f"{name}: index 0 received gradient")
                continue
        if gradient.abs().max().item() == 0.0 and "embeddings" not in name:
            problems.append(f"{name}: zero gradient")
    return len(problems) == 0, "all parameters receive finite gradient; index-0 rows receive none" if not problems else "; ".join(problems[:5])


def test_loss(context):
    model = context["make_model"](0)
    inputs, targets = context["make_batch"](8, 0)
    with torch.no_grad():
        logits = model(inputs)
        ours = model_module.compute_loss(logits, targets).item()
        log_probabilities = torch.log_softmax(logits, dim=-1)
        manual = -log_probabilities.gather(-1, targets.unsqueeze(-1)).mean().item()
        entropy = -(log_probabilities.exp() * log_probabilities).sum(-1).mean().item()
    # At initialisation the model must be uninformative: its predictive entropy should be
    # (almost) the maximum possible, ln 15. The loss itself can differ from ln 15, because
    # small random class preferences shared by all events may disfavour frequent classes.
    passed = abs(ours - manual) < 1e-5 and entropy > math.log(15) - 0.05
    return passed, (f"loss {ours:.5f} = manual {manual:.5f}; mean predictive entropy {entropy:.4f} "
                    f"(maximum ln 15 = {math.log(15):.4f})")


def test_valid_mask(context):
    model = context["make_model"](0)
    inputs, targets = context["make_batch"](8, 0)
    valid_mask = torch.ones_like(targets, dtype=torch.bool)
    valid_mask[:, :5] = False
    random_targets = targets.clone()
    generator = torch.Generator(device="cpu").manual_seed(0)
    random_targets[:, :5] = torch.randint(0, 15, (targets.shape[0], 5), generator=generator).to(targets.device)
    with torch.no_grad():
        logits = model(inputs)
        loss_original = model_module.compute_loss(logits, targets, valid_mask).item()
        loss_random = model_module.compute_loss(logits, random_targets, valid_mask).item()
    return loss_original == loss_random, f"loss with original targets {loss_original:.6f}; with randomised masked targets {loss_random:.6f}"


def test_memorise_one_batch(context):
    model = context["make_model"](0)
    inputs, targets = context["make_batch"](4, 0)
    optimiser = torch.optim.AdamW(model.parameters(), lr=3e-3, weight_decay=0.0)
    loss_value = None
    for step in range(400):
        loss = model_module.compute_loss(model(inputs), targets)
        optimiser.zero_grad()
        loss.backward()
        optimiser.step()
        loss_value = loss.item()
        if loss_value < 0.01:
            break
    return loss_value < 0.05, f"training loss on one batch of 4 windows after {step + 1} steps: {loss_value:.4f}"


def test_shuffled_labels(context):
    """Trained on permuted targets, the model must not beat the class prior on validation."""
    model = context["make_model"](0)
    train_batches = context["shuffled_label_batches"]
    optimiser = torch.optim.AdamW(model.parameters(), lr=1e-3)
    for inputs, targets in train_batches:
        loss = model_module.compute_loss(model(inputs), targets)
        optimiser.zero_grad()
        loss.backward()
        optimiser.step()
    inputs, targets = context["validation_batch"]
    with torch.no_grad():
        validation_nll = model_module.compute_loss(model(inputs), targets).item()
    prior_nll = context["prior_nll_on_validation_batch"]
    tolerance = context["shuffled_label_tolerance"]
    passed = validation_nll >= prior_nll - tolerance
    return passed, (f"validation NLL after training on shuffled labels {validation_nll:.4f}; "
                    f"class-prior NLL {prior_nll:.4f} (pass if not lower by more than {tolerance})")


def test_target_side_columns_cannot_change_inputs(context):
    events = context["events"]
    encoding = context["encoding"]
    encode = context["encode_features"]
    features_original, _ = encode(events, encoding)
    perturbed = events.copy()
    rng = np.random.default_rng(0)
    perturbed["action_id"] = rng.integers(0, 15, len(events))
    perturbed["end_zone"] = rng.integers(0, 96, len(events))
    perturbed["action"] = "SHOT"
    perturbed["type_name"] = "Shot"
    features_perturbed, _ = encode(perturbed, encoding)
    unchanged = all(np.array_equal(features_original[name], features_perturbed[name]) for name in features_original)
    return unchanged, "all 19 input arrays identical after randomising action_id, action, type_name, end_zone"


def test_index_zero_stays_zero(context):
    model = context["make_model"](0)
    optimiser = torch.optim.AdamW(model.parameters(), lr=1e-2, weight_decay=0.1)
    for step in range(5):
        inputs, targets = context["make_batch"](8, step * 8)
        loss = model_module.compute_loss(model(inputs), targets)
        optimiser.zero_grad()
        loss.backward()
        optimiser.step()
    largest = 0.0
    for feature in model_module.IDENTITY_FEATURES:
        if feature in model.embedding.embeddings:
            largest = max(largest, model.embedding.embeddings[feature].weight[0].abs().max().item())
    return largest == 0.0, f"largest |value| in any index-0 identity row after 5 AdamW steps (weight decay 0.1): {largest}"


ALL_TESTS = [
    ("1 shapes", test_shapes),
    ("2 causal mask", test_causal_mask),
    ("3 attention vs reference", test_attention_matches_reference),
    ("4 attention probabilities", test_attention_probabilities),
    ("5 residual additions", test_residual_additions),
    ("6 LayerNorm placement", test_layernorm_placement),
    ("7 positional embeddings", test_positional_embeddings),
    ("8 output dimension", test_output_dimension),
    ("9 determinism", test_determinism),
    ("10 gradient flow", test_gradient_flow),
    ("11 loss", test_loss),
    ("12 valid-position mask", test_valid_mask),
    ("13 memorise one batch", test_memorise_one_batch),
    ("14 shuffled labels", test_shuffled_labels),
    ("15 target-side columns", test_target_side_columns_cannot_change_inputs),
    ("16 index 0 stays zero", test_index_zero_stays_zero),
]


def run_all(context):
    results = []
    for name, test in ALL_TESTS:
        try:
            passed, detail = test(context)
        except Exception as error:          # a crashing test is a failed test
            passed, detail = False, f"ERROR: {error!r}"
        results.append({"test": name, "passed": bool(passed), "detail": detail})
    return results

Writing tests/test_model.py


### 5.1 Setting up the test context

Most tests use real validation windows. Test 14 needs a small training run on **shuffled labels**: the targets of 2,000 training windows are randomly permuted (breaking any link between input and answer), the model is trained briefly, and its validation NLL is compared with the NLL of simply predicting the training class frequencies. A correct pipeline cannot do better than that prior, because nothing in the input relates to the shuffled answers. The tolerance of 0.01 nats is a numerical allowance for the finite validation sample, not a research threshold.

The following cells check the working directory and the Python import path in the Colab environment, so that the test module in `tests/` can be imported. They do not affect the tests themselves.

In [12]:
import os

print("Current directory:", os.getcwd())

print("\nFiles here:")
print(os.listdir("."))

print("\nTests directory exists:", os.path.exists("tests"))

if os.path.exists("tests"):
    print("Tests contents:", os.listdir("tests"))

Current directory: /content/drive/MyDrive/soccer_mi/track_a

Files here:
['data', 'artifacts', 'src', 'tests', 'results', 'runs', 'figures', 'config', 'notebooks']

Tests directory exists: True
Tests contents: ['test_model.py']


In [13]:
import sys
import os
import importlib

PROJECT_ROOT = "/content/drive/MyDrive/soccer_mi/track_a"

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

importlib.invalidate_caches()

print(sys.path[:3])

['/content/drive/MyDrive/soccer_mi/track_a', '/content', '/env/python']


In [16]:
from pathlib import Path

tests_dir = Path("/content/drive/MyDrive/soccer_mi/track_a/tests")
(tests_dir / "__init__.py").touch()

print(list(tests_dir.iterdir()))

[PosixPath('/content/drive/MyDrive/soccer_mi/track_a/tests/test_model.py'), PosixPath('/content/drive/MyDrive/soccer_mi/track_a/tests/__init__.py')]


In [17]:
import sys
import importlib

PROJECT_ROOT = "/content/drive/MyDrive/soccer_mi/track_a"

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# Remove any previously imported conflicting "tests" package
for name in list(sys.modules):
    if name == "tests" or name.startswith("tests."):
        del sys.modules[name]

importlib.invalidate_caches()

In [18]:
import tests.test_model as model_tests
importlib.reload(model_tests)

torch.use_deterministic_algorithms(True, warn_only=True)
train_windows = data.build_windows(events, 32, 8, split="train")

# Test 14: shuffled-label training batches (targets permuted across the whole subset)
generator = np.random.default_rng(0)
subset_windows = train_windows[generator.choice(len(train_windows), size=2000, replace=False)]
subset_rows = torch.as_tensor(subset_windows, device=DEVICE)
subset_inputs, subset_targets = encoded.batch(subset_rows)
permutation = torch.as_tensor(generator.permutation(subset_targets.numel()), device=DEVICE)
shuffled_targets = subset_targets.reshape(-1)[permutation].reshape(subset_targets.shape)
shuffled_label_batches = []
for epoch in range(2):
    for start in range(0, 2000, 64):
        batch_inputs = {name: tensor[start:start + 64] for name, tensor in subset_inputs.items()}
        shuffled_label_batches.append((batch_inputs, shuffled_targets[start:start + 64]))

validation_inputs, validation_targets = make_batch(400, 0)
train_class_counts = np.bincount(targets[events["split"].to_numpy() == "train"], minlength=15)
train_prior = train_class_counts / train_class_counts.sum()
prior_nll = float(-np.log(train_prior[validation_targets.cpu().numpy()]).mean())

test_context = {
    "make_model": make_model,
    "make_batch": make_batch,
    "shuffled_label_batches": shuffled_label_batches,
    "validation_batch": (validation_inputs, validation_targets),
    "prior_nll_on_validation_batch": prior_nll,
    "shuffled_label_tolerance": 0.01,
    "events": events,
    "encoding": encoding,
    "encode_features": data.encode_features,
}

In [19]:
start_time = time.time()
test_results = model_tests.run_all(test_context)
elapsed = time.time() - start_time
test_table = pd.DataFrame(test_results).set_index("test")
print(f"{test_table['passed'].sum()} of {len(test_table)} tests passed in {elapsed:.0f} s on {DEVICE}")

with open(RESULTS_DIR / "03_test_results.json", "w") as handle:
    json.dump({"device": DEVICE, "torch": torch.__version__, "results": test_results}, handle, indent=2)
pd.set_option("display.max_colwidth", 140)
test_table

16 of 16 tests passed in 27 s on cpu


,passed,detail
test,,
1 shapes,True,"observed {'logits': (8, 32, 15), 'attention_probabilities': (8, 4, 32, 32), 'resid_post': (8, 32, 128)}"
2 causal mask,True,max change at positions <= 10: 0.0; after: 0.756
3 attention vs reference,True,max |ours - reference| = 7.45e-09
4 attention probabilities,True,max |row sum - 1| = 2.38e-07; max future weight = 0.0
5 residual additions,True,max reconstruction error = 0.00e+00
6 LayerNorm placement,True,sub-layer inputs equal LN(residual): error 0.0e+00; explicit formula (x-mean)/sqrt(var+eps)*scale+shift: error 4.8e-07
7 positional embeddings,True,output change when positions removed 0.188; position gradient norm 0.197
8 output dimension,True,classes = 15 (15 actions; END is not a target)
9 determinism,True,params identical True; outputs identical True; 10-step losses identical True


**RESULT.** The table lists each test, whether it passed, and the measured quantity behind the verdict.

**How to read the details.**

* *Causal mask*: replacing every event after position 10 changes nothing at positions 0–10 (difference exactly 0) but does change later positions — the model cannot see the future, and the test is sensitive enough to detect change where change is allowed.
* *Attention vs reference*: the hand-written attention and PyTorch's built-in causal attention agree to within floating-point rounding.
* *Residual additions / LayerNorm*: each intermediate tensor is recomputed from its parts; the errors are at rounding level.
* *Loss*: the loss matches a manual −log softmax computation exactly (2.69291). At initialisation the model's mean predictive entropy is 2.6839, close to the maximum ln 15 ≈ 2.7081, so the model has no real preference among classes before training.
* *LayerNorm*: the sub-layer inputs match an explicit (x − mean)/√(variance + ε) computation. (Their variance is not exactly 1, because the ε = 1e-5 term matters when the residual vectors are as small as they are at initialisation.)
* *Memorise one batch*: the loss on 4 windows falls to 0.0094 after 56 steps, so the model is able to fit data.
* *Shuffled labels*: with the link between inputs and answers broken, validation NLL (1.8653) stays at the class-prior level (1.8618), so no hidden path from the target to the input exists.
* *Index 0*: the fixed identity rows remain exactly zero, even under weight decay.

**SCIENTIFIC SIGNIFICANCE.** These tests show that the code computes the intended decoder-only, causal, pre-LayerNorm Transformer. Later results can be interpreted as properties of that architecture rather than of an implementation error.

**LIMITATION.** The tests check the computation, not whether the architecture is a good choice; that is decided by the experiments in notebooks 04–05. Determinism is verified on the device used here; GPU kernels can differ from CPU kernels at rounding level, which is why seeds are also recorded for every run.